# E-Commerce Intelligence & Decision Support Platform
## Stage 1: Data Preprocessing & Cleaning

**Dataset**: UCI Online Retail II (`online_retail_II.xlsx`)  
**Target User**: E-Commerce Manager / Decision Maker  

In this notebook, we follow simple, step-by-step beginner code to clean the raw data:
1. **Step 1**: Import basic libraries
2. **Step 2**: Load the two Excel sheets (`Year 2009-2010` and `Year 2010-2011`)
3. **Step 3**: Standardize column names and combine sheets into one DataFrame
4. **Step 4**: Check dataset info, data types, and missing values
5. **Step 5**: Check and remove duplicate rows
6. **Step 6**: Identify and separate cancellations (Invoices starting with `'C'`)
7. **Step 7**: Remove invalid rows (negative quantity or zero/negative price)
8. **Step 8**: Convert `InvoiceDate` to datetime and calculate `TotalAmount = Quantity * UnitPrice`
9. **Step 9**: Create customer dataset and customer summary
10. **Step 10**: Create product performance summary
11. **Step 11**: Save cleaned data to CSV files


In [ ]:

import pandas as pd
import numpy as np
import os

print("Libraries imported successfully!")

Libraries imported successfully!


### Step 2: Load the Two Excel Sheets
The dataset comes in an Excel file with two sheets:
- `Year 2009-2010`
- `Year 2010-2011`

We read each sheet into its own dataframe.


In [ ]:

file_name = "online_retail_II.xlsx"
if not os.path.exists(file_name):
    file_name = os.path.join("data", "raw", "online_retail_II.xlsx")

print("Loading Sheet 1: Year 2009-2010...")
df1 = pd.read_excel(file_name, sheet_name="Year 2009-2010")
print("Sheet 1 Shape:", df1.shape)

print("Loading Sheet 2: Year 2010-2011...")
df2 = pd.read_excel(file_name, sheet_name="Year 2010-2011")
print("Sheet 2 Shape:", df2.shape)

Loading Sheet 1: Year 2009-2010...
Sheet 1 Shape: (525461, 8)
Loading Sheet 2: Year 2010-2011...
Sheet 2 Shape: (541910, 8)


### Step 3: Standardize Column Names and Combine
We rename the columns to match standard naming:
- `Invoice` -> `InvoiceNo`
- `Price` -> `UnitPrice`
- `Customer ID` -> `CustomerID`

Then we combine both years into one dataframe called `df`.


In [ ]:

columns_to_rename = {
    'Invoice': 'InvoiceNo',
    'Price': 'UnitPrice',
    'Customer ID': 'CustomerID'
}

df1 = df1.rename(columns=columns_to_rename)
df2 = df2.rename(columns=columns_to_rename)

df = pd.concat([df1, df2], ignore_index=True)

print("Combined DataFrame Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Combined DataFrame Shape: (1067371, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Step 4: Check Data Types and Missing Values
Let us inspect the columns, data types, and count of missing values (`NaN`).


In [ ]:

print("--- Column Data Types ---")
print(df.dtypes)

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

--- Column Data Types ---
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

--- Missing Values Count ---
InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     243007
Country             0
dtype: int64


### Step 5: Check and Remove Duplicate Rows
Exact duplicate rows can artificially inflate sales numbers, so we remove them.


In [ ]:

num_duplicates = df.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

Number of duplicate rows: 34335
Shape after dropping duplicates: (1033036, 8)


### Step 6: Identify and Separate Cancellations
Invoices that start with the letter **'C'** (like `C489449`) represent **cancellations**.
Their quantities are negative.

We keep cancellations in a separate dataframe `df_cancellations` so we can analyse return rates later, without distorting our positive sales numbers.


In [ ]:

df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()


is_cancellation = df['InvoiceNo'].str.startswith('C') | (df['Quantity'] < 0)


df_cancellations = df[is_cancellation].copy()
df_sales = df[~is_cancellation].copy()

print("Total cancellations:", len(df_cancellations))
print("Total active sales:", len(df_sales))

Total cancellations: 22497
Total active sales: 1010539


### Step 7: Handle Invalid Quantities and Prices
For normal sales transactions:
- `Quantity` must be greater than 0.
- `UnitPrice` must be greater than 0.

Any rows with zero or negative price/quantity are filtered out.


In [ ]:

df_clean = df_sales[(df_sales['Quantity'] > 0) & (df_sales['UnitPrice'] > 0)].copy()

print("Clean valid sales count:", len(df_clean))

Clean valid sales count: 1007913


### Step 8: Convert `InvoiceDate` and Create `TotalAmount`
We convert `InvoiceDate` to datetime format, and calculate:
$$\text{TotalAmount} = \text{Quantity} \times \text{UnitPrice}$$


In [ ]:

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])


df_clean['TotalAmount'] = (df_clean['Quantity'] * df_clean['UnitPrice']).round(2)

print("Date range:", df_clean['InvoiceDate'].min(), "to", df_clean['InvoiceDate'].max())
print(f"Total Gross Revenue: £{df_clean['TotalAmount'].sum():,.2f}")
df_clean.head()

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Total Gross Revenue: £20,476,260.43


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


### Step 9: Customer Dataset & Customer Summary
For customer recommendations and segmentation:
- We filter rows where `CustomerID` is known (not null).
- We group by `CustomerID` to calculate total spend, total orders, total units, and unique products.


In [ ]:

df_customer_transactions = df_clean[df_clean['CustomerID'].notnull()].copy()
df_customer_transactions['CustomerID'] = df_customer_transactions['CustomerID'].astype(int)


customer_summary = df_customer_transactions.groupby('CustomerID').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'StockCode': 'nunique'
}).reset_index()


customer_summary.columns = ['CustomerID', 'TotalSpend', 'TotalOrders', 'TotalUnits', 'UniqueProducts']
customer_summary['TotalSpend'] = customer_summary['TotalSpend'].round(2)

print("Customer transaction lines:", len(df_customer_transactions))
print("Unique customers count:", len(customer_summary))
customer_summary.head()

Customer transaction lines: 779425
Unique customers count: 5878


,CustomerID,TotalSpend,TotalOrders,TotalUnits,UniqueProducts
0,12346,77556.46,12,74285,27
1,12347,4921.53,8,2967,126
2,12348,2019.40,5,2714,25
3,12349,4428.69,4,1624,138
4,12350,334.40,1,197,17


### Step 10: Product Performance Summary
We aggregate sales by `StockCode` to find total units sold, total revenue, and unique customer count.


In [ ]:

product_summary = df_clean.groupby('StockCode').agg({
    'Description': 'first',
    'Quantity': 'sum',
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).reset_index()

product_summary.columns = ['StockCode', 'Description', 'TotalQuantitySold', 'TotalRevenue', 'TotalInvoices', 'UniqueCustomers']
product_summary['TotalRevenue'] = product_summary['TotalRevenue'].round(2)

product_summary = product_summary.sort_values(by='TotalRevenue', ascending=False)

print("Unique products count:", len(product_summary))
print("\nTop 5 Products by Revenue:")
product_summary.head()

Unique products count: 4917

Top 5 Products by Revenue:


,StockCode,Description,TotalQuantitySold,TotalRevenue,TotalInvoices,UniqueCustomers
4902,M,Manual,9629,339226.24,784,438
1513,22423,REGENCY CAKESTAND 3 TIER,26478,330590.32,3918,1314
4901,DOT,DOTCOM POSTAGE,1415,309854.11,1415,1
4456,85123A,WHITE HANGING HEART T-LIGHT HOLDER,94203,257724.71,5365,1490
4441,85099B,JUMBO BAG RED WHITE SPOTTY,96757,180569.34,3989,978


### Step 11: Save Cleaned Data to CSV Files
We save all the cleaned datasets into `data/processed/` so they can be loaded easily in the next EDA step without reloading the Excel file.


In [ ]:

os.makedirs("data/processed", exist_ok=True)

df_clean.to_csv("data/processed/cleaned_transactions.csv", index=False)
print("1. Saved data/processed/cleaned_transactions.csv")

df_customer_transactions.to_csv("data/processed/customer_transactions.csv", index=False)
print("2. Saved data/processed/customer_transactions.csv")

df_cancellations.to_csv("data/processed/cancellations.csv", index=False)
print("3. Saved data/processed/cancellations.csv")

customer_summary.to_csv("data/processed/customer_data.csv", index=False)
print("4. Saved data/processed/customer_data.csv")

product_summary.to_csv("data/processed/product_data.csv", index=False)
print("5. Saved data/processed/product_data.csv")

print("\n Data Preprocessing & Cleaning is Complete!")

1. Saved data/processed/cleaned_transactions.csv
2. Saved data/processed/customer_transactions.csv
3. Saved data/processed/cancellations.csv
4. Saved data/processed/customer_data.csv
5. Saved data/processed/product_data.csv

 Data Preprocessing & Cleaning is Complete!
